# SNN FC-gradient spike reconstruction toolkit

This notebook contains the updated consolidated implementation for your two-stage SNN reconstruction workflow.

It includes:

1. rank diagnostics for each layer: `rank(G)`, `rank(S)`, `rank(S_tilde)`, `rank(grad_W)`, and `rank([grad_W | grad_b])`;
2. coordinate compression for Stage 1 using three optional mechanisms:
   - zero-gradient coordinates: fixed to 0;
   - bias-like / all-one coordinates: fixed to 1;
   - duplicate gradient columns: tied to one representative and restored later;
3. candidate selection by `selection_mode="residual"` or `selection_mode="global"`;
4. layer selection: reconstruct only `"x0"`, only `"s1"`, or any subset such as `["x0", "s1", "s2"]`;
5. generalized ordering: choose a target layer and choose which downstream layers constrain the ordering.

The code assumes a fully connected SNN with the layer mapping:

- `fc1` recovers `x0`;
- `fc2` recovers `s1`;
- `fc3` recovers `s2`;
- `fc4` recovers `s3`.


In [ ]:
# If running fresh (e.g. Colab), uncomment:
# !pip install torch snntorch --quiet
import random
import time
import torch
import torch.nn as nn
import snntorch as snn
from snntorch import surrogate
import numpy as np
from scipy.optimize import milp, LinearConstraint, Bounds
from scipy.sparse import hstack, eye, csc_matrix
import numpy as np
from scipy.optimize import milp, LinearConstraint, Bounds
from scipy.sparse import hstack, eye, csc_matrix
torch.manual_seed(0)
device = torch.device("cpu")   # single example, CPU is fine

# ----- configuration -----
T          = 32        # number of simulation timesteps
M_in       = 2312       # number of input (event) channels  -> this is "M" in the attack
H1, H2, H3 = 1024, 512, 256   # hidden layer widths
n_classes  = 100       # output classes
beta       = 0.9       # LIF membrane leak (decay)
U_thr      = 1.0       # firing threshold (snntorch default)
spike_grad = surrogate.atan()   # arctan surrogate gradient

print(f"T={T}, M_in={M_in}, widths=({H1},{H2},{H3}), classes={n_classes}, beta={beta}")

In [ ]:
class FCSNN(nn.Module):
    def __init__(self):
        super().__init__()
        # 4 fully connected layers, all with bias
        self.fc1  = nn.Linear(M_in, H1, bias=True)
        self.lif1 = snn.Leaky(beta=beta, threshold=U_thr, spike_grad=spike_grad)
        self.fc2  = nn.Linear(H1, H2, bias=True)
        self.lif2 = snn.Leaky(beta=beta, threshold=U_thr, spike_grad=spike_grad)
        self.fc3  = nn.Linear(H2, H3, bias=True)
        self.lif3 = snn.Leaky(beta=beta, threshold=U_thr, spike_grad=spike_grad)
        self.fc4  = nn.Linear(H3, n_classes, bias=True)
        self.lif4 = snn.Leaky(beta=beta, threshold=U_thr, spike_grad=spike_grad)

    def forward(self, x_seq):
        dtype = x_seq.dtype
        device = x_seq.device

        m1 = self.lif1.init_leaky().to(device=device, dtype=dtype)
        m2 = self.lif2.init_leaky().to(device=device, dtype=dtype)
        m3 = self.lif3.init_leaky().to(device=device, dtype=dtype)
        m4 = self.lif4.init_leaky().to(device=device, dtype=dtype)

        spk_out_sum = 0

        rec = {"x0": [], "s1": [], "s2": [], "s3": [], "s4": []}

        for t in range(x_seq.shape[0]):
            x_t = x_seq[t].to(dtype=dtype)

            cur1 = self.fc1(x_t)
            s1, m1 = self.lif1(cur1, m1)
            s1 = s1.to(dtype=dtype)
            m1 = m1.to(dtype=dtype)

            cur2 = self.fc2(s1)
            s2, m2 = self.lif2(cur2, m2)
            s2 = s2.to(dtype=dtype)
            m2 = m2.to(dtype=dtype)

            cur3 = self.fc3(s2)
            s3, m3 = self.lif3(cur3, m3)
            s3 = s3.to(dtype=dtype)
            m3 = m3.to(dtype=dtype)

            cur4 = self.fc4(s3)
            s4, m4 = self.lif4(cur4, m4)
            s4 = s4.to(dtype=dtype)
            m4 = m4.to(dtype=dtype)

            spk_out_sum = spk_out_sum + s4

            rec["x0"].append(x_t.detach())
            rec["s1"].append(s1.detach())
            rec["s2"].append(s2.detach())
            rec["s3"].append(s3.detach())
            rec["s4"].append(s4.detach())

        rec = {k: torch.stack(v, dim=0) for k, v in rec.items()}

        return spk_out_sum, rec
def init_snn_weights(net, w_gain=4.0, b_val=0.3):
    """Scale weights up and add small positive bias so LIF neurons actually fire."""
    for layer in [net.fc1, net.fc2, net.fc3, net.fc4]:
        # larger-variance weights: fan-in scaled, then amplified by w_gain
        nn.init.kaiming_uniform_(layer.weight, a=0.0)
        with torch.no_grad():
            layer.weight.mul_(w_gain)          # amplify so current can cross U_thr
            layer.bias.fill_(b_val)            # small positive bias -> encourages firing
    return net

net = FCSNN().to(device).double()
net = init_snn_weights(net, w_gain=4.0, b_val=0.1)

print(net)
print("\nfc1 has bias:", net.fc1.bias is not None,
      "| fc4 has bias:", net.fc4.bias is not None)


# Use double precision
net = net.double()


criterion = torch.nn.CrossEntropyLoss()

In [ ]:
import tonic
import tonic.transforms as transforms

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

sensor_size = tonic.datasets.NMNIST.sensor_size  # (34, 34, 2)

frame_transform = transforms.Compose([
    transforms.Denoise(filter_time=10000),
    transforms.ToFrame(sensor_size=sensor_size, n_time_bins=T),
])

dataset = tonic.datasets.NMNIST(
    save_to="./data",
    train=True,
    transform=frame_transform,
)

# --- pick one sample ---

sample_index = 50
frames, label = dataset[sample_index]                    # frames: (T, 2, 34, 34) numpy, label: int

x = torch.tensor(frames, dtype=torch.float64, device=device)   # (T, 2, 34, 34)
y = torch.tensor([label], device=device)  
print(x.shape)                     # (1,)

# print(f"x: {tuple(x.shape)}   dtype={x.dtype}")
# print(f"y: {y.item()}")
print(f"total spikes in sample: {int(x.sum().item())}")

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
sample_index = 20512
frames, label = dataset[sample_index]                    # frames: (T, 2, 34, 34) numpy, label: int
frames = (frames > 0).astype("float64")  # binary spike frames
x = torch.tensor(frames, dtype=torch.float64, device=device)   # (T, 2, 34, 34)
y = torch.tensor([label], device=device)  
x_np = x.detach().cpu().numpy()   # (T, 2, 34, 34)

# --- View 1: composite (sum over time, ON=red, OFF=blue) --------------
on_map  = x_np[:, 1].sum(0)      # ON events per pixel
off_map = x_np[:, 0].sum(0)      # OFF events per pixel

rgb = np.zeros((34, 34, 3), dtype=np.float64)
rgb[..., 0] = on_map  / max(on_map.max(),  1)   # red channel
rgb[..., 2] = off_map / max(off_map.max(), 1)   # blue channel

plt.figure(figsize=(3, 3))
plt.imshow(rgb)
plt.title(f"N-MNIST #{sample_index}   label = {y.item()}")
plt.axis("off")
plt.show()

# --- View 2: grid of all T time bins (ON − OFF) -----------------------
T = x_np.shape[0]
cols = 8
rows = (T + cols - 1) // cols

fig, axes = plt.subplots(rows, cols, figsize=(cols * 1.2, rows * 1.2))
vmax = np.abs(x_np[:, 1] - x_np[:, 0]).max()

for t, ax in enumerate(axes.flat):
    if t < T:
        frame = x_np[t, 1] - x_np[t, 0]      # ON minus OFF
        ax.imshow(frame, cmap="seismic", vmin=-vmax, vmax=vmax)
        ax.set_title(f"t={t}", fontsize=7)
    ax.axis("off")

plt.suptitle(f"label = {y.item()}", y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
sample_index = 20512
frames, label = dataset[sample_index]                    # frames: (T, 2, 34, 34) numpy, label: int
frames = (frames > 0).astype("float64")  # binary spike frames
x = torch.tensor(frames, dtype=torch.float64, device=device)   # (T, 2, 34, 34)
y = torch.tensor([label], device=device)                       # (1,)



# Convert model to float64
net = net.to(device).double()

# Prepare input as binary float64
frames, label = dataset[sample_index]
frames = (frames > 0).astype("float64")

x = torch.tensor(frames, dtype=torch.float64, device=device)
x = x.view(x.shape[0], 1, -1)   # [T, 1, M]

# Label must stay long
y = torch.tensor([label], dtype=torch.long, device=device)

criterion = nn.CrossEntropyLoss()

# Check dtypes
print("x dtype:", x.dtype)
print("fc1 weight dtype:", net.fc1.weight.dtype)
print("y dtype:", y.dtype)

net.zero_grad(set_to_none=True)
out, rec = net(x)
loss = criterion(out, y)
loss.backward()
for name in ["x0", "s1", "s2", "s3", "s4"]:
    rate = rec[name].float().mean().item()
    print(name, "spike rate:", rate)
# loss = torch.nn.CrossEntropyLoss()(out, y)

# net.zero_grad()
# loss.backward()

print("Loss:", loss.item())
# can=[4e-7,5e-7,6e-7,7e-7,8e-7,9e-7,1e-6,2e-6,3e-6,4e-6,5e-6,6e-6,7e-6,8e-6,9e-6,1e-5,2e-5,3e-5,4e-5,5e-5,6e-5,7e-5,8e-5,9e-5,1e-4,2e-4,3e-4,4e-4,5e-4,6e-4,7e-4,8e-4,9e-4]
# # Recover x0, s1, s2, s3 from gradients and verify them against rec
# for i in can:


In [ ]:
import time
from collections import Counter
import numpy as np
import torch
from scipy.optimize import milp, LinearConstraint, Bounds

## General utilities

In [ ]:
# ============================================================
# General utilities
# ============================================================

def _to_numpy(x):
    if torch.is_tensor(x):
        return x.detach().cpu().numpy()
    return np.asarray(x)


def _as_time_batch_matrix(x, device=None):
    """
    Convert candidate/time tensor to [K/T, 1, M].
    Accepts [T,M], [T,1,M], NumPy or torch.
    """
    if isinstance(x, np.ndarray):
        x = torch.tensor(x, dtype=torch.float32, device=device)
    elif torch.is_tensor(x):
        if device is not None:
            x = x.to(device)
    else:
        raise TypeError(f"Expected tensor or ndarray, got {type(x)}")

    if x.ndim == 2:
        x = x.unsqueeze(1)
    if x.ndim != 3:
        raise ValueError(f"Expected [T,M] or [T,1,M], got {tuple(x.shape)}")
    if x.shape[1] != 1:
        raise ValueError(f"Expected batch size 1, got {tuple(x.shape)}")
    return x


def _spike_key(x):
    """Convert one spike vector [1,M] or [M] into a hashable binary tuple."""
    x = x.detach().cpu() if torch.is_tensor(x) else torch.tensor(x)
    if x.ndim == 2:
        if x.shape[0] != 1:
            raise ValueError(f"Expected batch size 1, got {tuple(x.shape)}")
        x = x[0]
    return tuple((x > 0).to(torch.int64).tolist())


def _make_counter(candidate_tensor):
    """candidate_tensor shape [K,1,M]."""
    candidate_tensor = _as_time_batch_matrix(candidate_tensor)
    return Counter(_spike_key(candidate_tensor[k]) for k in range(candidate_tensor.shape[0]))


def candidate_tuple(z):
    return tuple(np.asarray(z).astype(int).tolist())


def candidates_to_spike_tensor(candidates, width, device, dtype=torch.float32):
    """Convert augmented candidates z=[spikes,1] into tensor [K,1,width]."""
    if len(candidates) == 0:
        return torch.empty((0, 1, width), dtype=dtype, device=device)
    X = np.stack([np.asarray(z[:-1]).astype(float) for z in candidates], axis=0)
    return torch.tensor(X, dtype=dtype, device=device).unsqueeze(1)


def make_S_tilde_from_rec(spike_rec):
    """
    Convert recorded spikes to augmented matrix S_tilde.

    Input:
        spike_rec shape [T,1,M] or [T,M]
    Output:
        S_tilde shape [M+1,T]
    """
    S = _to_numpy(spike_rec)

    if S.ndim == 3:
        if S.shape[1] != 1:
            raise ValueError(f"Expected batch size 1, got shape {S.shape}")
        S = S[:, 0, :]
    if S.ndim != 2:
        raise ValueError(f"Expected [T,M] or [T,1,M], got shape {S.shape}")

    S = S.T
    S = (S > 0).astype(int)
    bias = np.ones((1, S.shape[1]), dtype=int)
    return np.vstack([S, bias])


def true_columns_as_set(S_tilde_gt):
    return {
        tuple(S_tilde_gt[:, t].astype(int).tolist())
        for t in range(S_tilde_gt.shape[1])
    }


## Double-safe forward functions and rank diagnostics

In [ ]:
# ============================================================
# Double-safe forward functions and rank diagnostics
# ============================================================

def _init_mems(net, dtype, device):
    return (
        net.lif1.init_leaky().to(device=device, dtype=dtype),
        net.lif2.init_leaky().to(device=device, dtype=dtype),
        net.lif3.init_leaky().to(device=device, dtype=dtype),
        net.lif4.init_leaky().to(device=device, dtype=dtype),
    )


def _clone_mems(mems):
    return tuple(m.clone() if hasattr(m, "clone") else m for m in mems)


def forward_with_current_recording(net, x_seq):
    """
    Forward pass that records x0, s1..s4 and cur1..cur4 with retained gradients.
    Safe for float64/double precision.
    """
    dtype = x_seq.dtype
    device = x_seq.device

    m1, m2, m3, m4 = _init_mems(net, dtype=dtype, device=device)
    spk_out_sum = None

    rec_rank = {
        "x0": [], "s1": [], "s2": [], "s3": [], "s4": [],
        "cur1": [], "cur2": [], "cur3": [], "cur4": [],
    }

    for t in range(x_seq.shape[0]):
        x_t = x_seq[t].to(device=device, dtype=dtype)

        cur1 = net.fc1(x_t)
        cur1.retain_grad()
        s1, m1 = net.lif1(cur1, m1)
        s1 = s1.to(dtype=dtype)
        m1 = m1.to(dtype=dtype)

        cur2 = net.fc2(s1)
        cur2.retain_grad()
        s2, m2 = net.lif2(cur2, m2)
        s2 = s2.to(dtype=dtype)
        m2 = m2.to(dtype=dtype)

        cur3 = net.fc3(s2)
        cur3.retain_grad()
        s3, m3 = net.lif3(cur3, m3)
        s3 = s3.to(dtype=dtype)
        m3 = m3.to(dtype=dtype)

        cur4 = net.fc4(s3)
        cur4.retain_grad()
        s4, m4 = net.lif4(cur4, m4)
        s4 = s4.to(dtype=dtype)
        m4 = m4.to(dtype=dtype)

        spk_out_sum = s4 if spk_out_sum is None else spk_out_sum + s4

        rec_rank["x0"].append(x_t.detach())
        rec_rank["s1"].append(s1.detach())
        rec_rank["s2"].append(s2.detach())
        rec_rank["s3"].append(s3.detach())
        rec_rank["s4"].append(s4.detach())

        rec_rank["cur1"].append(cur1)
        rec_rank["cur2"].append(cur2)
        rec_rank["cur3"].append(cur3)
        rec_rank["cur4"].append(cur4)

    for key in ["x0", "s1", "s2", "s3", "s4"]:
        rec_rank[key] = torch.stack(rec_rank[key], dim=0)

    return spk_out_sum, rec_rank


def matrix_rank_np(A, tol=None):
    A = np.asarray(A)
    if tol is None:
        return np.linalg.matrix_rank(A)
    return np.linalg.matrix_rank(A, tol=tol)


def spike_matrix_from_time_tensor(x):
    """Convert [T,1,M] or [T,M] into S with shape [M,T]."""
    x_np = _to_numpy(x)
    if x_np.ndim == 3:
        if x_np.shape[1] != 1:
            raise ValueError(f"Expected batch size 1, got shape {x_np.shape}")
        x_np = x_np[:, 0, :]
    if x_np.ndim != 2:
        raise ValueError(f"Expected [T,M] or [T,1,M], got shape {x_np.shape}")
    return x_np.T


def gradient_matrix_from_cur_list(cur_list):
    """Convert list of cur.grad tensors into G with shape [N,T]."""
    grads = []
    for cur in cur_list:
        if cur.grad is None:
            raise RuntimeError("cur.grad is None. Did you call retain_grad() and backward()?")
        g = cur.grad.detach().cpu()
        if g.ndim == 2:
            if g.shape[0] != 1:
                raise ValueError(f"Expected batch size 1, got grad shape {tuple(g.shape)}")
            g = g[0]
        grads.append(g)
    return torch.stack(grads, dim=1).numpy()


def layer_rank_diagnostics(net, x, y, criterion, rank_tol=None, layers="all", print_results=True):
    """
    Compute rank(G), rank(S), rank(S_tilde), rank(grad_W), rank([grad_W|grad_b])
    for selected layers.

    layers can be "all", "x0", "s1", "s2", "s3", or list/tuple.
    """
    net.zero_grad(set_to_none=True)
    out, rec_rank = forward_with_current_recording(net, x)
    loss = criterion(out, y)
    loss.backward()

    all_specs = {
        "x0": {"layer_name": "fc1", "presynaptic_name": "x0", "cur_name": "cur1", "layer": net.fc1},
        "s1": {"layer_name": "fc2", "presynaptic_name": "s1", "cur_name": "cur2", "layer": net.fc2},
        "s2": {"layer_name": "fc3", "presynaptic_name": "s2", "cur_name": "cur3", "layer": net.fc3},
        "s3": {"layer_name": "fc4", "presynaptic_name": "s3", "cur_name": "cur4", "layer": net.fc4},
    }

    if layers == "all":
        selected = ["x0", "s1", "s2", "s3"]
    elif isinstance(layers, str):
        selected = [layers]
    else:
        selected = list(layers)

    results = {}

    for key in selected:
        if key not in all_specs:
            raise ValueError(f"Unknown layer {key}. Choose from {list(all_specs.keys())} or 'all'.")
        spec = all_specs[key]
        layer_name = spec["layer_name"]
        pre_name = spec["presynaptic_name"]
        cur_name = spec["cur_name"]
        layer = spec["layer"]

        G = gradient_matrix_from_cur_list(rec_rank[cur_name])
        S = spike_matrix_from_time_tensor(rec_rank[pre_name])
        S_tilde = np.vstack([S, np.ones((1, S.shape[1]), dtype=S.dtype)])

        grad_W = layer.weight.grad.detach().cpu().numpy()
        grad_b = layer.bias.grad.detach().cpu().numpy()
        M_tilde = np.concatenate([grad_W, grad_b[:, None]], axis=1)

        grad_W_hat = G @ S.T
        grad_b_hat = G @ np.ones((G.shape[1],), dtype=G.dtype)

        abs_W_err = np.linalg.norm(grad_W - grad_W_hat)
        rel_W_err = abs_W_err / (np.linalg.norm(grad_W) + 1e-12)
        abs_b_err = np.linalg.norm(grad_b - grad_b_hat)
        rel_b_err = abs_b_err / (np.linalg.norm(grad_b) + 1e-12)

        results[key] = {
            "layer_name": layer_name,
            "presynaptic": pre_name,
            "G_shape": G.shape,
            "S_shape": S.shape,
            "S_tilde_shape": S_tilde.shape,
            "grad_W_shape": grad_W.shape,
            "M_tilde_shape": M_tilde.shape,
            "rank_G": matrix_rank_np(G, tol=rank_tol),
            "rank_S": matrix_rank_np(S, tol=rank_tol),
            "rank_S_tilde": matrix_rank_np(S_tilde, tol=rank_tol),
            "rank_grad_W": matrix_rank_np(grad_W, tol=rank_tol),
            "rank_M_tilde": matrix_rank_np(M_tilde, tol=rank_tol),
            "factorization_abs_W_error": abs_W_err,
            "factorization_rel_W_error": rel_W_err,
            "bias_abs_error": abs_b_err,
            "bias_rel_error": rel_b_err,
        }

    if print_results:
        print("\nLayer rank diagnostics")
        print("======================")
        print("Loss:", float(loss.detach().cpu()))
        print("T:", x.shape[0])
        for key, r in results.items():
            print("\n" + "-" * 70)
            print(f"{r['layer_name']} recovers {r['presynaptic']}")
            print("-" * 70)
            print("G shape:        ", r["G_shape"])
            print("S shape:        ", r["S_shape"])
            print("S_tilde shape:  ", r["S_tilde_shape"])
            print("grad_W shape:   ", r["grad_W_shape"])
            print("M_tilde shape:  ", r["M_tilde_shape"])
            print()
            print("rank(G):        ", r["rank_G"])
            print("rank(S):        ", r["rank_S"])
            print("rank(S_tilde):  ", r["rank_S_tilde"])
            print("rank(grad_W):   ", r["rank_grad_W"])
            print("rank(M_tilde):  ", r["rank_M_tilde"])
            print()
            print("factorization rel W error:", f"{r['factorization_rel_W_error']:.3e}")
            print("bias rel error:           ", f"{r['bias_rel_error']:.3e}")

    return results, rec_rank, loss


## Augmented gradient, SVD, and projection

In [ ]:
# ============================================================
# Augmented gradient, coordinate compression, SVD, and projection
# ============================================================


def _close_vector(a, b, atol=1e-12, rtol=1e-10):
    """Robust vector closeness test using absolute + relative norm."""
    a = np.asarray(a, dtype=np.float64)
    b = np.asarray(b, dtype=np.float64)
    diff = np.linalg.norm(a - b)
    scale = max(np.linalg.norm(a), np.linalg.norm(b), 1.0)
    return diff <= atol + rtol * scale


def _split_duplicate_bucket(gW, bucket, atol=1e-12, rtol=1e-10):
    """
    Split a rounded bucket into truly close duplicate groups.
    This protects against accidental grouping due to rounding.
    """
    bucket = list(bucket)
    used = set()
    groups = []

    for j in bucket:
        if j in used:
            continue

        group = [j]
        used.add(j)

        for k in bucket:
            if k in used:
                continue
            if _close_vector(gW[:, j], gW[:, k], atol=atol, rtol=rtol):
                group.append(k)
                used.add(k)

        groups.append(np.array(group, dtype=int))

    return groups


def compress_input_coordinates_from_gradient(
    gW,
    gb,
    zero_tol=1e-12,
    one_atol=1e-12,
    one_rtol=1e-10,
    duplicate_decimals=12,
    use_zero_elimination=True,
    use_one_elimination=True,
    use_duplicate_elimination=True,
    verify_duplicates=True,
):
    """
    Compress input-neuron coordinates using gradient-column structure.

    For an FC layer, grad_W has shape [out_features, in_features].
    Each input neuron corresponds to one COLUMN of grad_W.

    Rules under rank(G)=T:
        grad_W[:, j] == 0       -> spike train j is all zeros.
        grad_W[:, j] == grad_b  -> spike train j is all ones.
        grad_W[:, j] == grad_W[:, k] -> spike trains j and k are identical.

    The three mechanisms are independently configurable:
        use_zero_elimination
        use_one_elimination
        use_duplicate_elimination
    """
    gW = np.asarray(gW, dtype=np.float64)
    gb = np.asarray(gb, dtype=np.float64)

    full_width = gW.shape[1]

    detected_zero = []
    detected_one = []
    other = []

    for j in range(full_width):
        col = gW[:, j]
        if np.linalg.norm(col) <= zero_tol:
            detected_zero.append(j)
        elif _close_vector(col, gb, atol=one_atol, rtol=one_rtol):
            detected_one.append(j)
        else:
            other.append(j)

    detected_zero = np.array(detected_zero, dtype=int)
    detected_one = np.array(detected_one, dtype=int)
    other = np.array(other, dtype=int)

    fixed_zero_idx = detected_zero if use_zero_elimination else np.array([], dtype=int)
    fixed_one_idx = detected_one if use_one_elimination else np.array([], dtype=int)

    # Coordinates not fixed as constants remain variables.
    variable_parts = [other]
    if not use_zero_elimination:
        variable_parts.append(detected_zero)
    if not use_one_elimination:
        variable_parts.append(detected_one)

    if len(variable_parts) == 0:
        variable_idx = np.array([], dtype=int)
    else:
        variable_idx = np.concatenate([p for p in variable_parts if len(p) > 0]).astype(int)
        variable_idx = np.sort(variable_idx)

    # Duplicate compression among variable coordinates.
    if use_duplicate_elimination:
        buckets = {}
        for j in variable_idx:
            key = np.round(gW[:, j], decimals=duplicate_decimals).tobytes()
            buckets.setdefault(key, []).append(int(j))

        groups = []
        for bucket in buckets.values():
            if verify_duplicates and len(bucket) > 1:
                groups.extend(_split_duplicate_bucket(gW, bucket, atol=one_atol, rtol=one_rtol))
            else:
                groups.append(np.array(bucket, dtype=int))
    else:
        groups = [np.array([j], dtype=int) for j in variable_idx]

    # Deterministic ordering of groups by representative index.
    groups = sorted(groups, key=lambda g: int(g[0]))
    rep_idx = np.array([int(g[0]) for g in groups], dtype=int)

    compression = {
        "full_width": full_width,
        "detected_zero_idx": detected_zero,
        "detected_one_idx": detected_one,
        "zero_idx": fixed_zero_idx,
        "one_idx": fixed_one_idx,
        "variable_idx": variable_idx,
        "groups": groups,
        "rep_idx": rep_idx,
        "num_detected_zero": len(detected_zero),
        "num_detected_one": len(detected_one),
        "num_fixed_zero": len(fixed_zero_idx),
        "num_fixed_one": len(fixed_one_idx),
        "num_variable_original": len(variable_idx),
        "num_representatives": len(rep_idx),
        "num_removed_by_duplicates": len(variable_idx) - len(rep_idx),
        "use_zero_elimination": use_zero_elimination,
        "use_one_elimination": use_one_elimination,
        "use_duplicate_elimination": use_duplicate_elimination,
        "zero_tol": zero_tol,
        "one_atol": one_atol,
        "one_rtol": one_rtol,
        "duplicate_decimals": duplicate_decimals,
    }

    return compression


def build_compressed_augmented_gradient(
    fc_layer,
    zero_tol=1e-12,
    one_atol=1e-12,
    one_rtol=1e-10,
    duplicate_decimals=12,
    use_zero_elimination=True,
    use_one_elimination=True,
    use_duplicate_elimination=True,
):
    """
    Build compressed augmented matrix:

        M_compressed = [grad_W[:, representatives] | grad_b]

    Constants and duplicate coordinates are restored after candidate recovery.
    """
    gW = fc_layer.weight.grad.detach().cpu().numpy().astype(np.float64)
    gb = fc_layer.bias.grad.detach().cpu().numpy().astype(np.float64)

    compression = compress_input_coordinates_from_gradient(
        gW=gW,
        gb=gb,
        zero_tol=zero_tol,
        one_atol=one_atol,
        one_rtol=one_rtol,
        duplicate_decimals=duplicate_decimals,
        use_zero_elimination=use_zero_elimination,
        use_one_elimination=use_one_elimination,
        use_duplicate_elimination=use_duplicate_elimination,
        verify_duplicates=True,
    )

    gW_compressed = gW[:, compression["rep_idx"]]
    M_compressed = np.concatenate([gW_compressed, gb[:, None]], axis=1).astype(np.float64)

    return M_compressed, compression, gW, gb


def expand_compressed_candidate(z_compressed, compression):
    """
    Expand compressed augmented candidate back to full dimension.

    z_compressed length = num_representatives + 1
    z_full length       = full_width + 1

    Expansion rules:
        zero_idx -> 0
        one_idx  -> 1
        duplicate group -> same bit as representative
        bias -> 1
    """
    full_width = compression["full_width"]
    z_full = np.zeros(full_width + 1, dtype=int)

    # Fixed constants.
    z_full[compression["zero_idx"]] = 0
    z_full[compression["one_idx"]] = 1

    # Representative and duplicate groups.
    for local_pos, group in enumerate(compression["groups"]):
        value = int(z_compressed[local_pos])
        z_full[group] = value

    # Phantom bias.
    z_full[-1] = int(z_compressed[-1])
    return z_full


# Backward-compatible aliases for older cells/code.
def build_reduced_augmented_gradient(fc_layer, zero_col_tol=1e-12, use_zero_elimination=True):
    M, compression, gW, gb = build_compressed_augmented_gradient(
        fc_layer=fc_layer,
        zero_tol=zero_col_tol,
        one_atol=1e-12,
        one_rtol=1e-10,
        duplicate_decimals=12,
        use_zero_elimination=use_zero_elimination,
        use_one_elimination=False,
        use_duplicate_elimination=False,
    )
    return M, compression["rep_idx"], compression["zero_idx"], gW, gb


def expand_reduced_candidate(z_reduced, active_idx, full_width):
    z_full = np.zeros(full_width + 1, dtype=int)
    z_full[active_idx] = np.asarray(z_reduced[:-1]).astype(int)
    z_full[-1] = int(z_reduced[-1])
    return z_full


def low_rank_factor_from_matrix(M_tilde, T, rank_mode="T", rank_tol=1e-10):
    M_tilde = np.asarray(M_tilde, dtype=np.float64)
    U, svals, Vt = np.linalg.svd(M_tilde, full_matrices=False)

    if rank_mode == "T":
        r = min(T, len(svals))
    elif rank_mode == "numerical":
        r = int(np.sum(svals > rank_tol))
        r = max(r, 1)
    else:
        raise ValueError("rank_mode must be 'T' or 'numerical'.")

    L = U[:, :r] @ np.diag(svals[:r])
    R = Vt[:r, :]
    return L, R, svals


def projection_residual(z, R):
    """For R from SVD, rows are orthonormal, so Pz=R.T@(R@z)."""
    z = np.asarray(z, dtype=np.float64)
    proj = R.T @ (R @ z)
    return np.linalg.norm(z - proj)


## Coordinate compression note

The reconstruction now supports three independent compression mechanisms: zero-coordinate fixing, one-coordinate fixing, and duplicate-column tying. These are controlled by `use_zero_elimination`, `use_one_elimination`, and `use_duplicate_elimination`.


## MILP candidate enumeration

In [ ]:
# ============================================================
# MILP candidate enumeration
# ============================================================

def solve_one_binary_candidate(R, previous_candidates=None, eps_eq=1e-6, time_limit=300.0):
    if previous_candidates is None:
        previous_candidates = []

    R = np.asarray(R, dtype=np.float64)
    r, d = R.shape
    num_vars = r + d

    c = np.zeros(num_vars)
    lb = np.concatenate([-np.inf * np.ones(r), np.zeros(d)])
    ub = np.concatenate([ np.inf * np.ones(r), np.ones(d)])
    bounds = Bounds(lb, ub)
    integrality = np.concatenate([np.zeros(r), np.ones(d)])

    constraints = []

    A_eq = np.zeros((d, num_vars))
    A_eq[:, :r] = R.T
    A_eq[:, r:] = -np.eye(d)
    constraints.append(LinearConstraint(A_eq, -eps_eq * np.ones(d), eps_eq * np.ones(d)))

    A_bias = np.zeros((1, num_vars))
    A_bias[0, r + d - 1] = 1.0
    constraints.append(LinearConstraint(A_bias, np.array([1.0]), np.array([1.0])))

    for z_prev in previous_candidates:
        z_prev = np.asarray(z_prev).astype(int)
        coeff = np.zeros(num_vars)
        num_ones = int(z_prev.sum())
        for j in range(d):
            coeff[r + j] = 1.0 if z_prev[j] == 0 else -1.0
        constraints.append(
            LinearConstraint(coeff.reshape(1, -1), np.array([1.0 - num_ones]), np.array([np.inf]))
        )

    result = milp(
        c=c,
        integrality=integrality,
        bounds=bounds,
        constraints=constraints,
        options={"time_limit": time_limit, "mip_rel_gap": 0.0},
    )

    if not result.success or result.x is None:
        return None, None, result

    h = result.x[:r]
    z = np.rint(result.x[r:]).astype(int)
    return h, z, result


def candidate_metrics(z, h, R):
    z_float = np.asarray(z, dtype=np.float64)
    v = R.T @ h
    eq_l2 = np.linalg.norm(v - z_float)
    eq_max = np.max(np.abs(v - z_float))
    proj = projection_residual(z_float, R)
    bin_res = np.sum(np.minimum(v**2, (v - 1.0)**2))
    bias_err = abs(v[-1] - 1.0)
    return {
        "proj": proj,
        "eq_l2": eq_l2,
        "eq_max": eq_max,
        "bin_res": bin_res,
        "bias_err": bias_err,
    }


def sort_candidate_pool(pool):
    return sorted(
        pool,
        key=lambda item: (
            item["metrics"]["proj"],
            item["metrics"]["eq_max"],
            item["metrics"]["eq_l2"],
            item["metrics"]["bin_res"],
            item["metrics"]["bias_err"],
        ),
    )


def enumerate_binary_candidates_ranked(
    R,
    top_k,
    max_pool=200,
    eps_eq=1e-6,
    eps_proj=np.inf,
    time_limit=300.0,
    verbose=True,
):
    """Generate a MILP candidate pool, filter by eps_proj, sort by residual, return top_k."""
    pool = []
    rejected_by_proj = []
    excluded_candidates = []
    start_time = time.perf_counter()
    certified_complete = False
    stop_reason = None

    for _ in range(max_pool):
        solve_start = time.perf_counter()
        h, z, result = solve_one_binary_candidate(
            R=R,
            previous_candidates=excluded_candidates,
            eps_eq=eps_eq,
            time_limit=time_limit,
        )
        solve_time = time.perf_counter() - solve_start
        elapsed = time.perf_counter() - start_time

        if z is None:
            stop_reason = result.message
            if result.status == 2:
                certified_complete = True
                stop_reason = "infeasible: no more candidates"
            if verbose:
                print("\nStopping candidate enumeration.")
                print("Solver success:", result.success)
                print("Solver status:", result.status)
                print("Solver message:", result.message)
            break

        excluded_candidates.append(z)
        metrics = candidate_metrics(z, h, R)
        item = {"z": z, "h": h, "metrics": metrics, "solve_time": solve_time, "elapsed": elapsed}

        if metrics["proj"] <= eps_proj:
            pool.append(item)
            if verbose:
                print(
                    f"stored {len(pool):03d} | "
                    f"proj={metrics['proj']:.3e} | "
                    f"eq_l2={metrics['eq_l2']:.3e} | "
                    f"eq_max={metrics['eq_max']:.3e} | "
                    f"bin={metrics['bin_res']:.3e} | "
                    f"bias={metrics['bias_err']:.3e} | "
                    f"solve={solve_time:.3f}s | elapsed={elapsed:.3f}s"
                )
        else:
            rejected_by_proj.append(item)
            if verbose:
                print(f"rejected by eps_proj | proj={metrics['proj']:.3e} > {eps_proj:.3e}")

    pool_sorted = sort_candidate_pool(pool)
    top_items = pool_sorted[:top_k]
    candidates = [item["z"] for item in top_items]
    hs = [item["h"] for item in top_items]
    total_time = time.perf_counter() - start_time

    if verbose:
        print("\nRanked candidate summary")
        print("------------------------")
        print("stored pool size:", len(pool_sorted))
        print("rejected by projection:", len(rejected_by_proj))
        print("top_k returned:", len(candidates))
        print("eps_eq:", eps_eq)
        print("eps_proj:", eps_proj)
        print("certified complete:", certified_complete)
        print("stop reason:", stop_reason)
        print(f"total time: {total_time:.3f}s")

    return {
        "candidates": candidates,
        "hs": hs,
        "top_items": top_items,
        "pool": pool_sorted,
        "rejected_by_proj": rejected_by_proj,
        "excluded_candidates": excluded_candidates,
        "certified_complete": certified_complete,
        "stop_reason": stop_reason,
        "time": total_time,
    }


## Global reconstruction candidate selection

In [ ]:
# ============================================================
# Global reconstruction candidate selection
# ============================================================

def score_candidate_set_reconstruction(M_tilde, candidates):
    if len(candidates) == 0:
        return {
            "rel_recon": np.inf, "abs_recon": np.inf,
            "bias_rel": np.inf, "bias_abs": np.inf,
            "rank_Z": 0, "G_hat": None, "M_hat": None,
        }

    Z = np.stack([np.asarray(z, dtype=np.float64) for z in candidates], axis=1)
    ZTZ_pinv = np.linalg.pinv(Z.T @ Z)
    G_hat = M_tilde @ Z @ ZTZ_pinv
    M_hat = G_hat @ Z.T

    abs_recon = np.linalg.norm(M_tilde - M_hat, ord="fro")
    rel_recon = abs_recon / (np.linalg.norm(M_tilde, ord="fro") + 1e-12)
    bias_true = M_tilde[:, -1]
    bias_hat = M_hat[:, -1]
    bias_abs = np.linalg.norm(bias_true - bias_hat)
    bias_rel = bias_abs / (np.linalg.norm(bias_true) + 1e-12)
    rank_Z = np.linalg.matrix_rank(Z)

    return {
        "rel_recon": rel_recon,
        "abs_recon": abs_recon,
        "bias_rel": bias_rel,
        "bias_abs": bias_abs,
        "rank_Z": rank_Z,
        "G_hat": G_hat,
        "M_hat": M_hat,
    }


def select_candidates_by_global_reconstruction(
    M_tilde,
    pool,
    top_k,
    candidate_limit=None,
    rank_penalty=10.0,
    verbose=True,
):
    """Greedy selection of candidates that jointly reconstruct [grad_W|grad_b]."""
    candidate_pool = pool[:candidate_limit] if candidate_limit is not None else pool
    selected_indices = []
    remaining_indices = list(range(len(candidate_pool)))
    history = []

    for step in range(top_k):
        best_idx = None
        best_score = np.inf
        best_diag = None

        for idx in remaining_indices:
            trial_indices = selected_indices + [idx]
            trial_candidates = [candidate_pool[i]["z"] for i in trial_indices]
            diag = score_candidate_set_reconstruction(M_tilde=M_tilde, candidates=trial_candidates)
            rank_deficit = max(0, len(trial_candidates) - diag["rank_Z"])
            score = diag["rel_recon"] + diag["bias_rel"] + rank_penalty * rank_deficit
            if score < best_score:
                best_score = score
                best_idx = idx
                best_diag = diag

        if best_idx is None:
            break

        selected_indices.append(best_idx)
        remaining_indices.remove(best_idx)
        history.append({
            "step": step + 1,
            "selected_pool_index": best_idx,
            "score": best_score,
            "rel_recon": best_diag["rel_recon"],
            "bias_rel": best_diag["bias_rel"],
            "rank_Z": best_diag["rank_Z"],
        })

        if verbose:
            print(
                f"global step {step+1:02d} | pool_idx={best_idx} | "
                f"rel_recon={best_diag['rel_recon']:.3e} | "
                f"bias_rel={best_diag['bias_rel']:.3e} | rank_Z={best_diag['rank_Z']}"
            )

    selected_items = [candidate_pool[i] for i in selected_indices]
    final_diag = score_candidate_set_reconstruction(
        M_tilde=M_tilde,
        candidates=[item["z"] for item in selected_items],
    )

    if verbose:
        print("\nGlobal reconstruction selection summary")
        print("--------------------------------------")
        print("selected:", len(selected_items))
        print("target top_k:", top_k)
        print("final rank_Z:", final_diag["rank_Z"])
        print("final rel_recon:", final_diag["rel_recon"])
        print("final bias_rel:", final_diag["bias_rel"])

    return selected_items, {
        "history": history,
        "final_diag": final_diag,
        "selected_indices": selected_indices,
    }


## Verification

In [ ]:
# ============================================================
# Verification
# ============================================================

def verify_pool_and_recovered_candidates(pool, recovered_candidates, S_tilde_gt, name="layer", print_examples=True):
    true_set = true_columns_as_set(S_tilde_gt)
    pool_set = {candidate_tuple(item["z"]) for item in pool}
    recovered_set = {candidate_tuple(z) for z in recovered_candidates}

    true_subset_of_pool = true_set.issubset(pool_set)
    recovered_subset_of_true = recovered_set.issubset(true_set)
    missing_true_from_pool = true_set - pool_set
    false_candidates_in_recovered = recovered_set - true_set
    true_recovered = recovered_set & true_set

    print("\n" + "-" * 70)
    print(f"Verification for {name}")
    print("-" * 70)
    print("Number of true unique vectors:", len(true_set))
    print("Number of whole-pool unique candidates:", len(pool_set))
    print("Number of recovered/top unique candidates:", len(recovered_set))
    print()
    print("Question 1: Are true vectors subset of the whole candidate pool?")
    print("Answer:", true_subset_of_pool)
    print()
    print("Question 2: Are recovered/top candidates subset of the true vectors?")
    print("Answer:", recovered_subset_of_true)
    print()
    print("True vectors recovered in top set:", len(true_recovered), "/", len(true_set))
    print("Missing true vectors from whole pool:", len(missing_true_from_pool))
    print("False candidates in recovered/top set:", len(false_candidates_in_recovered))

    if print_examples:
        if len(missing_true_from_pool) > 0:
            print("\nExample missing true vector from whole pool:")
            print(next(iter(missing_true_from_pool)))
        if len(false_candidates_in_recovered) > 0:
            print("\nExample false recovered/top candidate:")
            print(next(iter(false_candidates_in_recovered)))

    return {
        "true_subset_of_pool": true_subset_of_pool,
        "recovered_subset_of_true": recovered_subset_of_true,
        "num_true_unique": len(true_set),
        "num_pool_unique": len(pool_set),
        "num_recovered_unique": len(recovered_set),
        "num_true_recovered_in_top": len(true_recovered),
        "num_missing_true_from_pool": len(missing_true_from_pool),
        "num_false_candidates_in_recovered": len(false_candidates_in_recovered),
        "missing_true_from_pool": missing_true_from_pool,
        "false_candidates_in_recovered": false_candidates_in_recovered,
        "true_recovered": true_recovered,
    }


## Main reconstruction functions

In [ ]:
# ============================================================
# Main reconstruction functions
# ============================================================


def _compression_gt_diagnostics(S_tilde_gt, compression):
    """Check whether fixed/tied coordinates agree with ground truth, if available."""
    diagnostics = {}

    zero_idx = compression["zero_idx"]
    one_idx = compression["one_idx"]
    groups = compression["groups"]

    if len(zero_idx) > 0:
        zero_sum = int(S_tilde_gt[zero_idx, :].sum())
    else:
        zero_sum = 0

    if len(one_idx) > 0:
        one_violations = int(np.sum(S_tilde_gt[one_idx, :] != 1))
    else:
        one_violations = 0

    duplicate_violations = 0
    duplicate_groups_checked = 0

    for group in groups:
        if len(group) <= 1:
            continue
        duplicate_groups_checked += 1
        ref = S_tilde_gt[group[0], :]
        for coord in group[1:]:
            duplicate_violations += int(np.sum(S_tilde_gt[coord, :] != ref))

    diagnostics["zero_gt_sum"] = zero_sum
    diagnostics["one_gt_violations"] = one_violations
    diagnostics["duplicate_gt_violations"] = duplicate_violations
    diagnostics["duplicate_groups_checked"] = duplicate_groups_checked

    return diagnostics


def recover_input_spikes_from_layer(
    fc_layer,
    name,
    width,
    T,
    device,
    S_tilde_gt=None,
    top_k=None,
    max_pool=200,
    eps_eq=1e-6,
    eps_proj=np.inf,
    zero_col_tol=1e-12,
    one_atol=1e-12,
    one_rtol=1e-10,
    duplicate_decimals=12,
    use_zero_elimination=True,
    use_one_elimination=True,
    use_duplicate_elimination=True,
    time_limit=300.0,
    rank_mode="T",
    selection_mode="global",
    global_candidate_limit=100,
    rank_penalty=10.0,
    verbose=True,
):
    """
    Recover presynaptic spike candidates of one FC layer.

    Optional coordinate compression:
        use_zero_elimination=True      -> columns grad_W[:,j]≈0 are fixed to 0.
        use_one_elimination=True       -> columns grad_W[:,j]≈grad_b are fixed to 1.
        use_duplicate_elimination=True -> duplicate gradient columns are tied to one representative.

    selection_mode:
        'residual' -> select top_k candidates by individual residual.
        'global'   -> select top_k candidates by global reconstruction of compressed [grad_W | grad_b].
    """
    print("\n" + "=" * 80)
    print(f"Recovering {name}")
    print("=" * 80)

    M_tilde_compressed, compression, gW, gb = build_compressed_augmented_gradient(
        fc_layer=fc_layer,
        zero_tol=zero_col_tol,
        one_atol=one_atol,
        one_rtol=one_rtol,
        duplicate_decimals=duplicate_decimals,
        use_zero_elimination=use_zero_elimination,
        use_one_elimination=use_one_elimination,
        use_duplicate_elimination=use_duplicate_elimination,
    )

    print("Original weight-gradient shape:", gW.shape)
    print("Compressed augmented matrix shape:", M_tilde_compressed.shape)
    print("Compression settings:")
    print("  use_zero_elimination:", use_zero_elimination)
    print("  use_one_elimination:", use_one_elimination)
    print("  use_duplicate_elimination:", use_duplicate_elimination)
    print("Compression summary:")
    print("  detected zero coordinates:", compression["num_detected_zero"])
    print("  detected one coordinates:", compression["num_detected_one"])
    print("  fixed zero coordinates:", compression["num_fixed_zero"])
    print("  fixed one coordinates:", compression["num_fixed_one"])
    print("  variable original coordinates:", compression["num_variable_original"])
    print("  representatives kept:", compression["num_representatives"])
    print("  removed by duplicate tying:", compression["num_removed_by_duplicates"])

    if compression["num_fixed_zero"] > 0:
        print("  first fixed-zero positions:", compression["zero_idx"][:20])
    if compression["num_fixed_one"] > 0:
        print("  first fixed-one positions:", compression["one_idx"][:20])

    L, R, svals = low_rank_factor_from_matrix(M_tilde=M_tilde_compressed, T=T, rank_mode=rank_mode)
    rank_M = np.linalg.matrix_rank(M_tilde_compressed)

    print("rank(compressed M_tilde):", rank_M)
    print("R shape:", R.shape)
    print("target T:", T)
    if len(svals) >= T:
        print("smallest kept singular value:", svals[T - 1])
    if len(svals) > T:
        print("next singular value:", svals[T])

    if top_k is None:
        top_k = T

    enum = enumerate_binary_candidates_ranked(
        R=R,
        top_k=top_k,
        max_pool=max_pool,
        eps_eq=eps_eq,
        eps_proj=eps_proj,
        time_limit=time_limit,
        verbose=verbose,
    )

    if selection_mode == "residual":
        selected_items = enum["top_items"]
        global_selection = None
        if verbose:
            print("\nSelection mode: residual")
    elif selection_mode == "global":
        if verbose:
            print("\nSelection mode: global reconstruction")
        selected_items, global_selection = select_candidates_by_global_reconstruction(
            M_tilde=M_tilde_compressed,
            pool=enum["pool"],
            top_k=top_k,
            candidate_limit=global_candidate_limit,
            rank_penalty=rank_penalty,
            verbose=verbose,
        )
    else:
        raise ValueError("selection_mode must be 'residual' or 'global'.")

    compressed_candidates = [item["z"] for item in selected_items]
    hs = [item.get("h", None) for item in selected_items]

    full_candidates = [
        expand_compressed_candidate(z, compression)
        for z in compressed_candidates
    ]

    full_pool = []
    for idx, item in enumerate(enum["pool"]):
        item_full = dict(item)
        item_full["source_index"] = idx
        item_full["z_compressed"] = item["z"]
        item_full["z"] = expand_compressed_candidate(item["z"], compression)
        item_full["compression"] = compression
        full_pool.append(item_full)

    pool_verification = None
    compression_gt_diagnostics = None

    if S_tilde_gt is not None:
        compression_gt_diagnostics = _compression_gt_diagnostics(S_tilde_gt, compression)

        print("\nCompression ground-truth diagnostics")
        print("------------------------------------")
        print("zero fixed coordinates spike sum:", compression_gt_diagnostics["zero_gt_sum"])
        print("one fixed coordinates violations:", compression_gt_diagnostics["one_gt_violations"])
        print("duplicate groups checked:", compression_gt_diagnostics["duplicate_groups_checked"])
        print("duplicate-coordinate violations:", compression_gt_diagnostics["duplicate_gt_violations"])

        if compression_gt_diagnostics["zero_gt_sum"] != 0:
            print("WARNING: some fixed-zero coordinates are not zero in ground truth.")
        if compression_gt_diagnostics["one_gt_violations"] != 0:
            print("WARNING: some fixed-one coordinates are not one in ground truth.")
        if compression_gt_diagnostics["duplicate_gt_violations"] != 0:
            print("WARNING: some tied duplicate coordinates differ in ground truth.")

        pool_verification = verify_pool_and_recovered_candidates(
            pool=full_pool,
            recovered_candidates=full_candidates,
            S_tilde_gt=S_tilde_gt,
            name=name,
            print_examples=True,
        )

    dtype = next(fc_layer.parameters()).dtype if hasattr(fc_layer, "parameters") else torch.float32
    spikes = candidates_to_spike_tensor(full_candidates, width, device=device, dtype=dtype)

    return {
        "name": name,
        "spikes": spikes,
        "candidates_raw": full_candidates,
        "candidates_compressed": compressed_candidates,
        "hs": hs,
        "pool": full_pool,
        "pool_compressed": enum["pool"],
        "selected_items_compressed": selected_items,
        "compression": compression,
        "M_tilde_compressed": M_tilde_compressed,
        "M_tilde_reduced": M_tilde_compressed,  # backward-compatible alias
        "L": L,
        "R": R,
        "singular_values": svals,
        "rank_M_tilde": rank_M,
        "pool_verification": pool_verification,
        "compression_gt_diagnostics": compression_gt_diagnostics,
        "global_selection": global_selection,
        "certified_complete": enum["certified_complete"],
        "stop_reason": enum["stop_reason"],
        "time": enum["time"],
        "selection_mode": selection_mode,
        "eps_eq": eps_eq,
        "eps_proj": eps_proj,
        "use_zero_elimination": use_zero_elimination,
        "use_one_elimination": use_one_elimination,
        "use_duplicate_elimination": use_duplicate_elimination,
    }


def recover_selected_layers(
    net,
    rec,
    T,
    device,
    layers="all",
    top_k=None,
    max_pool=200,
    eps_eq=1e-6,
    eps_proj=np.inf,
    zero_col_tol=1e-12,
    one_atol=1e-12,
    one_rtol=1e-10,
    duplicate_decimals=12,
    use_zero_elimination=True,
    use_one_elimination=True,
    use_duplicate_elimination=True,
    time_limit=300.0,
    rank_mode="T",
    selection_mode="global",
    global_candidate_limit=100,
    rank_penalty=10.0,
    verbose=True,
):
    """
    Recover selected presynaptic layers.

    Available layer names:
        x0 -> fc1 gradients
        s1 -> fc2 gradients
        s2 -> fc3 gradients
        s3 -> fc4 gradients
    """
    layer_specs = {
        "x0": {"fc_layer": net.fc1, "width": net.fc1.in_features, "rec_spikes": rec["x0"]},
        "s1": {"fc_layer": net.fc2, "width": net.fc2.in_features, "rec_spikes": rec["s1"]},
        "s2": {"fc_layer": net.fc3, "width": net.fc3.in_features, "rec_spikes": rec["s2"]},
        "s3": {"fc_layer": net.fc4, "width": net.fc4.in_features, "rec_spikes": rec["s3"]},
    }

    if layers == "all":
        selected_layers = ["x0", "s1", "s2", "s3"]
    elif isinstance(layers, str):
        selected_layers = [layers]
    else:
        selected_layers = list(layers)

    for name in selected_layers:
        if name not in layer_specs:
            raise ValueError(f"Unknown layer '{name}'. Available: {list(layer_specs.keys())}")

    recovered = {}

    for name in selected_layers:
        spec = layer_specs[name]
        if verbose:
            print("\n" + "#" * 90)
            print(f"Selected layer: {name}")
            print("#" * 90)

        S_tilde_gt = make_S_tilde_from_rec(spec["rec_spikes"])

        recovered[name] = recover_input_spikes_from_layer(
            fc_layer=spec["fc_layer"],
            name=name,
            width=spec["width"],
            T=T,
            device=device,
            S_tilde_gt=S_tilde_gt,
            top_k=top_k if top_k is not None else T,
            max_pool=max_pool,
            eps_eq=eps_eq,
            eps_proj=eps_proj,
            zero_col_tol=zero_col_tol,
            one_atol=one_atol,
            one_rtol=one_rtol,
            duplicate_decimals=duplicate_decimals,
            use_zero_elimination=use_zero_elimination,
            use_one_elimination=use_one_elimination,
            use_duplicate_elimination=use_duplicate_elimination,
            time_limit=time_limit,
            rank_mode=rank_mode,
            selection_mode=selection_mode,
            global_candidate_limit=global_candidate_limit,
            rank_penalty=rank_penalty,
            verbose=verbose,
        )

    return recovered


## Candidate extraction and generalized ordering

In [ ]:
# ============================================================
# Candidate extraction and generalized ordering
# ============================================================

def _pool_sort_key(item):
    m = item.get("metrics", {})
    return (
        m.get("proj", m.get("full_proj", np.inf)),
        m.get("eq_max", np.inf),
        m.get("eq_l2", np.inf),
        m.get("bin_res", np.inf),
        m.get("bias_err", np.inf),
    )


def extract_candidates_from_recovered_layer(recovered_layer, source="spikes", device=None, sort_pool=True):
    """
    Extract candidates from one recovered layer.

    source:
        'spikes' -> recovered_layer['spikes']
        'pool'   -> all candidates in recovered_layer['pool']
        'raw'    -> recovered_layer['candidates_raw']
    """
    if torch.is_tensor(recovered_layer) or isinstance(recovered_layer, np.ndarray):
        X = _as_time_batch_matrix(recovered_layer, device=device).float()
        ids = list(range(X.shape[0]))
        items = [{"source_index": i} for i in ids]
        return X, ids, items

    if not isinstance(recovered_layer, dict):
        raise TypeError("recovered_layer must be a tensor, ndarray, or dictionary.")

    if source == "spikes":
        X = _as_time_batch_matrix(recovered_layer["spikes"], device=device).float()
        ids = list(range(X.shape[0]))
        items = [{"source_index": i} for i in ids]
        return X, ids, items

    if source == "pool":
        pool = list(recovered_layer["pool"])
        if sort_pool:
            pool = sorted(pool, key=_pool_sort_key)
        vectors, ids, items = [], [], []
        for i, item in enumerate(pool):
            z = np.asarray(item["z"]).astype(float)
            vectors.append(z[:-1])
            ids.append(item.get("source_index", i))
            items.append(item)
        X = torch.tensor(np.stack(vectors, axis=0), dtype=torch.float32, device=device).unsqueeze(1)
        return X, ids, items

    if source == "raw":
        raw = recovered_layer["candidates_raw"]
        vectors = [np.asarray(z).astype(float)[:-1] for z in raw]
        X = torch.tensor(np.stack(vectors, axis=0), dtype=torch.float32, device=device).unsqueeze(1)
        ids = list(range(X.shape[0]))
        items = [{"source_index": i} for i in ids]
        return X, ids, items

    raise ValueError("source must be 'spikes', 'pool', or 'raw'.")


_TARGET_ORDER = ["x0", "s1", "s2", "s3"]
_DOWNSTREAM = {
    "x0": ("s1", "s2", "s3", "s4"),
    "s1": ("s2", "s3", "s4"),
    "s2": ("s3", "s4"),
    "s3": ("s4",),
}


@torch.no_grad()
def _propagate_from_target_one_step(net, target_layer, x_t, mems):
    """
    Propagate one candidate from target_layer onward.

    target_layer:
        x0 -> candidate is input to fc1, produces s1,s2,s3,s4
        s1 -> candidate is input to fc2, produces s2,s3,s4
        s2 -> candidate is input to fc3, produces s3,s4
        s3 -> candidate is input to fc4, produces s4
    """
    dtype = next(net.parameters()).dtype
    x_t = x_t.to(device=next(net.parameters()).device, dtype=dtype)
    m1, m2, m3, m4 = mems
    produced = {}

    if target_layer == "x0":
        cur1 = net.fc1(x_t)
        s1, m1 = net.lif1(cur1, m1)
        s1 = s1.to(dtype=dtype); m1 = m1.to(dtype=dtype)
        produced["s1"] = s1.detach()

        cur2 = net.fc2(s1)
        s2, m2 = net.lif2(cur2, m2)
        s2 = s2.to(dtype=dtype); m2 = m2.to(dtype=dtype)
        produced["s2"] = s2.detach()

        cur3 = net.fc3(s2)
        s3, m3 = net.lif3(cur3, m3)
        s3 = s3.to(dtype=dtype); m3 = m3.to(dtype=dtype)
        produced["s3"] = s3.detach()

        cur4 = net.fc4(s3)
        s4, m4 = net.lif4(cur4, m4)
        s4 = s4.to(dtype=dtype); m4 = m4.to(dtype=dtype)
        produced["s4"] = s4.detach()

    elif target_layer == "s1":
        cur2 = net.fc2(x_t)
        s2, m2 = net.lif2(cur2, m2)
        s2 = s2.to(dtype=dtype); m2 = m2.to(dtype=dtype)
        produced["s2"] = s2.detach()

        cur3 = net.fc3(s2)
        s3, m3 = net.lif3(cur3, m3)
        s3 = s3.to(dtype=dtype); m3 = m3.to(dtype=dtype)
        produced["s3"] = s3.detach()

        cur4 = net.fc4(s3)
        s4, m4 = net.lif4(cur4, m4)
        s4 = s4.to(dtype=dtype); m4 = m4.to(dtype=dtype)
        produced["s4"] = s4.detach()

    elif target_layer == "s2":
        cur3 = net.fc3(x_t)
        s3, m3 = net.lif3(cur3, m3)
        s3 = s3.to(dtype=dtype); m3 = m3.to(dtype=dtype)
        produced["s3"] = s3.detach()

        cur4 = net.fc4(s3)
        s4, m4 = net.lif4(cur4, m4)
        s4 = s4.to(dtype=dtype); m4 = m4.to(dtype=dtype)
        produced["s4"] = s4.detach()

    elif target_layer == "s3":
        cur4 = net.fc4(x_t)
        s4, m4 = net.lif4(cur4, m4)
        s4 = s4.to(dtype=dtype); m4 = m4.to(dtype=dtype)
        produced["s4"] = s4.detach()

    else:
        raise ValueError(f"Unsupported target_layer {target_layer}.")

    return produced, (m1, m2, m3, m4)


def select_and_order_candidates(
    net,
    recovered,
    T,
    target_layer="x0",
    cover_layers=("s1",),
    target_source="pool",
    cover_source="spikes",
    sort_target_pool=True,
    device=None,
    true_rec=None,
    max_nodes=2_000_000,
    verbose=True,
):
    """
    Select and temporally order T candidates for target_layer from a candidate pool
    using downstream cover_layers.

    Examples:
        target_layer='x0', cover_layers=('s1','s2')
        target_layer='s1', cover_layers=('s2',)
        target_layer='s2', cover_layers=('s3','s4')

    true_rec is optional and used only for verification, not search.
    """
    if device is None:
        device = next(net.parameters()).device

    if target_layer not in _DOWNSTREAM:
        raise ValueError(f"target_layer must be one of {list(_DOWNSTREAM.keys())}")

    cover_layers = tuple(cover_layers)
    allowed = set(_DOWNSTREAM[target_layer])
    for layer in cover_layers:
        if layer not in allowed:
            raise ValueError(f"Cover layer {layer} is not downstream of {target_layer}. Allowed: {_DOWNSTREAM[target_layer]}")
        if layer not in recovered:
            raise ValueError(f"Cover layer {layer} not found in recovered dictionary.")

    if target_layer not in recovered:
        raise ValueError(f"Target layer {target_layer} not found in recovered dictionary.")

    target_candidates, target_ids, target_items = extract_candidates_from_recovered_layer(
        recovered_layer=recovered[target_layer],
        source=target_source,
        device=device,
        sort_pool=sort_target_pool,
    )
    K = target_candidates.shape[0]
    if K < T:
        raise ValueError(f"Target candidate pool has {K} candidates, but T={T}.")

    target_counters = {}
    for layer in cover_layers:
        X_layer, _, _ = extract_candidates_from_recovered_layer(
            recovered_layer=recovered[layer],
            source=cover_source,
            device=device,
            sort_pool=True,
        )
        target_counters[layer] = _make_counter(X_layer)
        if X_layer.shape[0] != T and verbose:
            print(f"Warning: cover layer {layer} has {X_layer.shape[0]} candidates, T={T}.")

    start_time = time.perf_counter()
    used = [False] * K
    selected_indices = []
    stats = {"nodes": 0, "candidate_tests": 0, "accepted_steps": 0, "backtracks": 0, "max_depth": 0}
    solution = None

    def counters_available(keys):
        return all(target_counters[layer][key] > 0 for layer, key in keys.items())

    def counters_decrement(keys):
        for layer, key in keys.items():
            target_counters[layer][key] -= 1

    def counters_increment(keys):
        for layer, key in keys.items():
            target_counters[layer][key] += 1

    def backtrack(t, mems):
        nonlocal solution
        stats["nodes"] += 1
        stats["max_depth"] = max(stats["max_depth"], t)
        if stats["nodes"] > max_nodes:
            return False
        if t == T:
            solution = list(selected_indices)
            return True

        for i in range(K):
            if used[i]:
                continue
            stats["candidate_tests"] += 1
            x_t = target_candidates[i]
            produced, new_mems = _propagate_from_target_one_step(
                net=net,
                target_layer=target_layer,
                x_t=x_t,
                mems=_clone_mems(mems),
            )
            keys = {layer: _spike_key(produced[layer]) for layer in cover_layers}
            if not counters_available(keys):
                continue

            used[i] = True
            selected_indices.append(i)
            counters_decrement(keys)
            stats["accepted_steps"] += 1

            if backtrack(t + 1, new_mems):
                return True

            counters_increment(keys)
            selected_indices.pop()
            used[i] = False
            stats["backtracks"] += 1

        return False

    dtype = next(net.parameters()).dtype
    mems0 = _init_mems(net, dtype=dtype, device=device)
    success = backtrack(0, mems0)
    elapsed = time.perf_counter() - start_time

    if success:
        ordered_target = target_candidates[solution].clone()
        selected_source_indices = [target_ids[i] for i in solution]
        selected_items = [target_items[i] for i in solution]
    else:
        ordered_target = None
        selected_source_indices = None
        selected_items = None

    verification = None
    if true_rec is not None and ordered_target is not None and target_layer in true_rec:
        true_target = _as_time_batch_matrix(true_rec[target_layer], device=device)
        exact_temporal_match = torch.equal((ordered_target > 0).cpu(), (true_target > 0).cpu())
        ordered_counter = _make_counter(ordered_target)
        true_counter = _make_counter(true_target)
        per_timestep_match = [
            bool(torch.equal((ordered_target[t] > 0).cpu(), (true_target[t] > 0).cpu()))
            for t in range(T)
        ]
        verification = {
            "exact_temporal_match": exact_temporal_match,
            "same_multiset_as_true": ordered_counter == true_counter,
            "matched_timesteps": sum(per_timestep_match),
            "T": T,
        }

    if verbose:
        print("\nCandidate selection and temporal ordering")
        print("-----------------------------------------")
        print("target_layer:", target_layer)
        print("cover_layers:", cover_layers)
        print("target_source:", target_source)
        print("cover_source:", cover_source)
        print("target candidate pool size:", K)
        print("T:", T)
        print("success:", success)
        print("nodes visited:", stats["nodes"])
        print("candidate tests:", stats["candidate_tests"])
        print("accepted steps:", stats["accepted_steps"])
        print("backtracks:", stats["backtracks"])
        print("max depth:", stats["max_depth"])
        print(f"time: {elapsed:.4f}s")
        if success:
            print("\nSelected indices in sorted target pool:")
            print(solution)
            print("\nSelected source indices:")
            print(selected_source_indices)
        if verification is not None:
            print("\nVerification")
            print("------------")
            print("same multiset as true:", verification["same_multiset_as_true"])
            print("exact temporal match:", verification["exact_temporal_match"])
            print("matched timesteps:", verification["matched_timesteps"], "/", verification["T"])

    return {
        "success": success,
        "target_layer": target_layer,
        "ordered": ordered_target,
        "ordered_candidates": ordered_target,
        "selected_indices_in_sorted_pool": solution,
        "selected_source_indices": selected_source_indices,
        "selected_items": selected_items,
        "cover_layers": cover_layers,
        "target_pool_size": K,
        "stats": stats,
        "verification": verification,
        "time": elapsed,
    }


# Backward-compatible alias for your previous x0 function.
def select_and_order_x0_candidates(
    net,
    recovered,
    T,
    cover_layers=("s1",),
    x0_source="pool",
    hidden_source="spikes",
    sort_x0_pool=True,
    device=None,
    true_rec=None,
    max_nodes=2_000_000,
    verbose=True,
):
    return select_and_order_candidates(
        net=net,
        recovered=recovered,
        T=T,
        target_layer="x0",
        cover_layers=cover_layers,
        target_source=x0_source,
        cover_source=hidden_source,
        sort_target_pool=sort_x0_pool,
        device=device,
        true_rec=true_rec,
        max_nodes=max_nodes,
        verbose=verbose,
    )


## Usage examples

The following cells are examples. Adapt `eps_eq`, `eps_proj`, `max_pool`, and `layers` according to the current experiment.


In [ ]:
# Rank diagnostics for all layers
# Requires: net, x, y, criterion already defined

rank_results, rec_rank, loss_rank = layer_rank_diagnostics(
    net=net,
    x=x,
    y=y,
    criterion=criterion,
    layers="all",
    rank_tol=None,
    print_results=True,
)


In [ ]:
# Reconstruct selected layers
# Examples for layers: "x0", ["s1", "s2"], ["x0", "s1", "s2", "s3"], or "all"
# Compression controls:
#   use_zero_elimination=True/False
#   use_one_elimination=True/False
#   use_duplicate_elimination=True/False
recovered_x0 = recover_selected_layers(
    net=net,
    rec=rec,
    T=x.shape[0],
    device=device,
    layers=["x0", "s1", "s2"],
    top_k=x.shape[0],
    max_pool=1000,

    eps_eq=1e-9,
    eps_proj=1e-8,

    zero_col_tol=1e-12,
    one_atol=1e-13,
    one_rtol=1e-10,
    duplicate_decimals=12,

    use_zero_elimination=True,
    use_one_elimination=True,
    use_duplicate_elimination=True,

    time_limit=3000.0,
    rank_mode="T",
    selection_mode="residual",
    verbose=True,
)



In [ ]:
# Select and order x0 candidates using recovered downstream layers
# This uses recovered["x0"]["pool"] and recovered hidden-layer spikes.
# true_rec=rec is optional and only used for verification.

result_x0 = select_and_order_candidates(
    net=net,
    recovered=recovered,
    T=x.shape[0],
    target_layer="x0",
    cover_layers=("s1", "s2"),
    target_source="pool",
    cover_source="spikes",
    true_rec=rec,
    max_nodes=2_000_000,
    verbose=True,
)


In [ ]:
# Example: select/order s1 using s2 as the downstream constraint

result_s1 = select_and_order_candidates(
    net=net,
    recovered=recovered,
    T=x.shape[0],
    target_layer="s1",
    cover_layers=("s2",),
    target_source="pool",
    cover_source="spikes",
    true_rec=rec,
    max_nodes=2_000_000,
    verbose=True,
)


In [ ]:
# Optional: show original and reconstructed x0 side by side
# Assumes result_x0["ordered_x0"] exists and input shape is T x 2 x 34 x 34.

import matplotlib.pyplot as plt

def event_rgb(tensor, T, H=34, W=34):
    arr = tensor.detach().cpu().numpy() if torch.is_tensor(tensor) else np.asarray(tensor)
    arr = arr.reshape(T, 2, H, W)
    on_map = arr[:, 1].sum(0)
    off_map = arr[:, 0].sum(0)
    rgb = np.zeros((H, W, 3), dtype=np.float32)
    rgb[..., 0] = on_map / max(on_map.max(), 1)
    rgb[..., 2] = off_map / max(off_map.max(), 1)
    return rgb

recovered_spikes=result_x0["ordered_x0"]
original_rgb = event_rgb(x, T=x.shape[0])
recovered_rgb = event_rgb(recovered_spikes, T=x.shape[0])

plt.figure(figsize=(6, 3))
plt.subplot(1, 2, 1)
plt.imshow(original_rgb)
plt.title("Original")
plt.axis("off")

plt.subplot(1, 2, 2)
plt.imshow(recovered_rgb)
plt.title("Reconstructed")
plt.axis("off")

plt.tight_layout()
plt.show()


In [ ]:
def diagnose_true_vectors_against_layer(
    fc_layer,
    rec_spikes,
    T,
    use_zero_elimination=True,
    use_one_elimination=False,
    use_duplicate_elimination=False,
    zero_col_tol=1e-12,
    one_atol=1e-12,
    one_rtol=1e-10,
    duplicate_decimals=12,
    rank_mode="T",
):
    """
    Diagnostic:
    For each true vector, compute how close it is to the row space used by the solver.
    """

    M_tilde, compression, gW, gb = build_compressed_augmented_gradient(
        fc_layer=fc_layer,
        zero_tol=zero_col_tol,
        one_atol=one_atol,
        one_rtol=one_rtol,
        duplicate_decimals=duplicate_decimals,
        use_zero_elimination=use_zero_elimination,
        use_one_elimination=use_one_elimination,
        use_duplicate_elimination=use_duplicate_elimination,
    )

    L, R, svals = low_rank_factor_from_matrix(
        M_tilde=M_tilde,
        T=T,
        rank_mode=rank_mode,
    )

    S_tilde_full = make_S_tilde_from_rec(rec_spikes)

    # Compress true vectors according to the same compression.
    true_compressed = []

    for t in range(S_tilde_full.shape[1]):
        z_full = S_tilde_full[:, t].astype(int)

        vals = []

        for group in compression["groups"]:
            vals.append(z_full[group[0]])

            # sanity check duplicate group consistency
            if not np.all(z_full[group] == z_full[group[0]]):
                print("WARNING: true vector violates duplicate group at timestep", t)

        vals.append(1)  # bias
        true_compressed.append(np.array(vals, dtype=int))

    rows = []
    A = R.T

    for t, z in enumerate(true_compressed):
        h_star, _, _, _ = np.linalg.lstsq(A, z.astype(float), rcond=None)
        v = A @ h_star
        diff = v - z

        rows.append({
            "t": t,
            "eps_eq_needed": float(np.max(np.abs(diff))),
            "eps_proj_needed": float(np.linalg.norm(diff)),
            "spike_count_compressed": int(z[:-1].sum()),
        })

    import pandas as pd
    df = pd.DataFrame(rows)

    print("\nTrue-vector feasibility diagnostic")
    print("----------------------------------")
    print("M_tilde shape:", M_tilde.shape)
    print("rank(M_tilde):", np.linalg.matrix_rank(M_tilde))
    print("R shape:", R.shape)

    print("max eps_eq needed:", df["eps_eq_needed"].max())
    print("median eps_eq needed:", df["eps_eq_needed"].median())
    print("max eps_proj needed:", df["eps_proj_needed"].max())
    print("median eps_proj needed:", df["eps_proj_needed"].median())

    return df, R, compression

In [ ]:
df_x0_diag, R_x0, comp_x0 = diagnose_true_vectors_against_layer(
    fc_layer=net.fc1,
    rec_spikes=rec["x0"],
    T=x.shape[0],
    use_zero_elimination=True,
    use_one_elimination=False,
    use_duplicate_elimination=False,
    zero_col_tol=1e-12,
    rank_mode="T",
)

In [ ]:
df_x0_diag, R_x0, comp_x0 = diagnose_true_vectors_against_layer(
    fc_layer=net.fc1,
    rec_spikes=rec["x0"],
    T=x.shape[0],
    use_zero_elimination=True,
    use_one_elimination=False,
    use_duplicate_elimination=True,
    zero_col_tol=1e-12,
    rank_mode="T",
)